In [1]:
"""
-------------------------------------------------------------------------------
HDR BATCHER (UNIVERSAL RAW SUPPORT)
-------------------------------------------------------------------------------
Author:       Brian Willis
Assisted by:  Gemini AI
License:      MIT License 

DESCRIPTION:
Batch processes image brackets into HDRs.
Supports: JPG, TIF, PNG, and RAW formats (Canon CR3, Sony ARW, Adobe DNG).
Method: Uses 'rawpy' for developing Raw data and OpenCV Mertens Fusion for HDR.

PREREQUISITES:
- pip install rawpy opencv-python numpy
-------------------------------------------------------------------------------
"""

import os
import sys
import glob
import time
import cv2
import numpy as np
import tkinter as tk
from tkinter import filedialog

# Check for rawpy library
try:
    import rawpy
except ImportError:
    print("Missing library! Please run:")
    print("pip install rawpy")
    sys.exit()

def load_image_safe(file_path):
    """
    Smart loader: Handles both standard images and Raw files.
    Returns: BGR numpy array (ready for OpenCV) or None.
    """
    ext = os.path.splitext(file_path)[1].lower()
    
    # 1. Handle RAW files (CR3, ARW, DNG, NEF, ORF)
    if ext in ['.cr3', '.arw', '.dng', '.nef', '.orf']:
        try:
            print(f"    Developing Raw: {os.path.basename(file_path)}...")
            with rawpy.imread(file_path) as raw:
                # Postprocess: Develop raw data to RGB
                # use_camera_wb=True: Use the camera's white balance setting
                # no_auto_bright=True: Prevent auto-leveling (important for HDR brackets)
                rgb = raw.postprocess(use_camera_wb=True, no_auto_bright=True)
                
                # Convert RGB (Standard) to BGR (OpenCV format)
                bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
                return bgr
        except Exception as e:
            print(f"    [ERROR] Raw decode failed: {os.path.basename(file_path)} ({e})")
            return None

    # 2. Handle Standard files (JPG, TIF, PNG)
    else:
        return cv2.imread(file_path)

def merge_hdr(image_list, output_path):
    """
    Merges a list of loaded images using Mertens Fusion.
    """
    if len(image_list) < 2:
        print("    [SKIP] Need at least 2 valid images to merge.")
        return

    print("    Merging (Mertens Fusion)...")
    
    # Create the merger
    merge_mertens = cv2.createMergeMertens()
    
    # Generate the fusion
    res_mertens = merge_mertens.process(image_list)
    
    # Convert result (0-1 float) to 8-bit (0-255) for saving
    res_8bit = np.clip(res_mertens * 255, 0, 255).astype('uint8')
    
    print(f"    Saving: {os.path.basename(output_path)}")
    cv2.imwrite(output_path, res_8bit)

def main():
    print("--- HDR BATCHER (UNIVERSAL) ---")
    
    # 1. Select Folder
    root = tk.Tk()
    root.withdraw()
    print("Select the folder containing your bracketed images...")
    folder = filedialog.askdirectory(title="Select Input Folder")
    if not folder: return

    # 2. Find Files
    # Look for both standard and RAW formats
    extensions = ['*.tif', '*.tiff', '*.jpg', '*.jpeg', '*.png', '*.cr3', '*.arw', '*.dng']
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(folder, ext)))
        files.extend(glob.glob(os.path.join(folder, ext.upper())))
    
    files = sorted(list(set(files)))
    if not files:
        print("No images found.")
        return

    print(f"Found {len(files)} total files.")

    # 3. GROUPING LOGIC
    # Assumes files are named like "Site1_01.jpg", "Site1_02.jpg"
    # We group by removing the last part after the underscore.
    
    groups = {}
    for f in files:
        filename = os.path.basename(f)
        name_no_ext = os.path.splitext(filename)[0]
        
        # Logic: If filename has an underscore, split at the last one.
        # Example: "IMG_1234_1.CR3" -> Group ID: "IMG_1234"
        if '_' in name_no_ext:
            group_id = name_no_ext.rsplit('_', 1)[0]
        else:
            # Fallback: if no underscore, treat as single file (won't merge)
            group_id = name_no_ext 
            
        if group_id not in groups:
            groups[group_id] = []
        groups[group_id].append(f)

    print(f"Detected {len(groups)} bracket sets.")
    
    # 4. Process Each Group
    processed_count = 0
    start_time = time.time()
    
    for i, (group_name, file_paths) in enumerate(groups.items()):
        if len(file_paths) < 2:
            continue
            
        print(f"\n[Group {i+1}/{len(groups)}] {group_name} ({len(file_paths)} shots)")
        
        # Load images for this group
        loaded_imgs = []
        for fp in file_paths:
            img = load_image_safe(fp)
            if img is not None:
                loaded_imgs.append(img)
        
        # Merge if we successfully loaded brackets
        if loaded_imgs:
            output_filename = f"HDR_{group_name}.jpg"
            output_full_path = os.path.join(folder, output_filename)
            merge_hdr(loaded_imgs, output_full_path)
            processed_count += 1
            
            # Free memory immediately
            del loaded_imgs

    elapsed = (time.time() - start_time) / 60
    print(f"\n--- BATCH COMPLETE ---")
    print(f"Created {processed_count} HDRs in {elapsed:.1f} minutes.")
    input("Press Enter to exit...")

if __name__ == "__main__":
    main()

--- HDR BATCH PROCESSOR STARTING ---
Please select your INPUT folder from the popup window...
No images found in C:/Users/Willi/Desktop/HDR TST/Stop
